In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, make_scorer
import numpy as np



# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)



In [ ]:

print(df.head())





In [ ]:

print(df.info())



In [ ]:
print(df.describe())



In [ ]:


# Drop missing values from target column
Delivery_Time = df["Delivery_Time"].dropna()

# Plot target distribution safely
plt.figure(figsize=(6,4))
plt.hist(Delivery_Time, bins=30)
plt.xlabel("Delivery Time (minutes)")
plt.ylabel("Frequency")
plt.title("Delivery Time Distribution")
plt.show()



In [ ]:

# 1. Drop the 'Order_ID' column
df = df.drop(columns=["Order_ID"])




In [ ]:
# 2. Handle missing values
# Check missing values
df.isnull().sum()

# Fill numerical missing values with median
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical missing values with mode
cat_cols = df.select_dtypes(include=["object"]).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])



In [ ]:
# Task 3: Write your code here:
# 3. Check and remove duplicates
df = df.drop_duplicates()



In [ ]:
# Task 4: Write your code here:
# 4. Encode categorical variables (One-Hot Encoding)
df = pd.get_dummies(df, drop_first=True)



In [ ]:


# 5. Apply feature scaling (StandardScaler)
X = df.drop("Delivery_Time", axis=1)
y = df["Delivery_Time"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)





In [ ]:

# 6. Check target imbalance (Regression → usually not required)
# This step is not applicable for regression problems

In [ ]:


# 1. Split features and target
X = X_scaled
y = y




In [ ]:
# 2. K-Fold (Regression → NOT Stratified)
kf = KFold(n_splits=5, shuffle=False)

# 3. Train RandomForest model
model = RandomForestRegressor(n_estimators=200, random_state=42)

# 4. Evaluate using MAE ONLY
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

scores = cross_val_score(
    model,
    X,
    y,
    cv=kf,
    scoring=mae_scorer
)

# 5. Print averaged score
print("Average MAE:", -scores.mean())

In [ ]:
# Train model on full data
model.fit(X, y)

# 1. Plot feature importance
importances = model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(8,6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), np.array(df.drop("Delivery_Time", axis=1).columns)[indices])
plt.xlabel("Feature Importance")
plt.title("Feature Importance from RandomForest")
plt.show()


In [ ]:
# 2. Plot predicted delivery time histogram
y_pred = model.predict(X)

plt.figure(figsize=(6,4))
plt.hist(y_pred, bins=30)
plt.xlabel("Predicted Delivery Time")
plt.ylabel("Frequency")
plt.title("Predicted Delivery Time Distribution")
plt.show()


In [ ]:
# Task Bonus: Write your code here: